In [1]:
from pathlib import Path
from tqdm.notebook import tqdm
import shutil
import utils2p
import yaml
import numpy as np

from rye2p.thorimage.settings import FunctionalImagingSettings

# set upstairs or downstairs microscope
microscope = 'downstairs-2p'

prj = 'odor_space_collab'
# set project directory paths
if prj == 'natural_mixtures':
    PRJ_DATA_DIR = Path("/local/matrix/Remy-Data/projects/natural_mixtures")
    RAW_DATA_DIR = Path("/local/tensor-archive/Remy/natural_mixtures/raw_data")
    NAS_PROC_DIR = PRJ_DATA_DIR.joinpath('processed_data')
elif prj == 'haoru_rotation':
    PRJ_DATA_DIR = Path("/local/tensor/Haoru/rotation_project")
    RAW_DATA_DIR = Path("/local/tensor/Haoru/rotation_project/raw_data")
    NAS_PROC_DIR = Path("/local/tensor/Haoru/rotation_project/processed_data")
elif prj == 'odor_space_collab':
    PRJ_DATA_DIR = Path("/local/tensor-archive/Remy/odor_space_collab")
    RAW_DATA_DIR = Path("/local/tensor-archive/Remy/odor_space_collab/raw_data")
    NAS_PROC_DIR = Path("/local/matrix/Remy-Data/projects/odor_space_collab/processed_data")

In [17]:
folder = Path("/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15")

meta_files = sorted(list(folder.rglob("Experiment.xml")))
meta_files = [item for item in meta_files if 'tsub' not in str(item)]
meta_files

[PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/1/megamat1_001/Experiment.xml'),
 PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/1/megamat1_single_plane/Experiment.xml'),
 PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/1/megamat1_single_plane_001/Experiment.xml'),
 PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/2/megamat1/Experiment.xml'),
 PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/2/megamat1_001/Experiment.xml'),
 PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/2/megamat1_single_plane/Experiment.xml'),
 PosixPath('/local/matrix/Remy-Data/projects/odor_space_collab/processed_data/2023-04-15/2/megamat1_single_plane_001/Experiment.xml')]

In [18]:
for meta_file in meta_files:
    thorimage_settings = FunctionalImagingSettings.from_xml(meta_file)

    # Save ThorImage settings to proc_mov_dir
    thorimage_settings.to_yaml(meta_file.with_name('thorimage_settings.yaml'))

In [4]:
ts_file = Path("/local/tensor/Remy/natural_mixtures/processed_data/2022-02-11/1/kiwi_ea_eb_only/timestamps.npy")
timestamps = np.load(ts_file, allow_pickle=True).item()

thorimage_settings = FunctionalImagingSettings.from_yaml(ts_file.with_name('thorimage_settings.yaml'))


In [5]:
thorimage_settings = thorimage_settings.to_dict()

In [6]:
for k, v in timestamps.items():
    print(f"{k}: {v.size}")

display(timestamps['scope_ict'])
display(timestamps['scope_fct'])


frame_times: 101421
stack_times: 1690
olf_ict: 57
olf_fct: 58
scope_ict: 1
scope_fct: 1


array([17.1])

array([17.1])

In [25]:
scope_fct = timestamps['scope_fct']
scope_ict = timestamps['scope_ict']

raw_frame_times = timestamps['frame_times']

In [ ]:
# Validate counts against Experiment.xml - promoted into the library
# (roadmap step 1). Warns on mismatch; strict=True raises (the default
# flips to strict in roadmap step 2). thorimage_settings may be the
# FunctionalImagingSettings object or its to_dict() form.
from rye2p.thorsync.timestamps import validate_frame_counts

validate_frame_counts(raw_frame_times, scope_ict, scope_fct,
                      thorimage_settings, scope_padding=0.0)